# LibTorch Bucket DataLoader — Proxy Benchmark (Colab)

Builds and benchmarks the **length-bucketing LibTorch dataloader**
(`sft_libtorch_only`) and reports a **proxy training time** so it is directly
comparable to the category-scheduler pipeline.

**Proxy:** step time is assumed ∝ padded tokens the GPU processes, so
`proxy = alpha · padded_tokens + beta · batches`. With `alpha=1, beta=0` (default)
this is the padded-token count — the same unit both pipelines print, so you can
compare them one-to-one. Set `PROXY_ALPHA`/`PROXY_BETA` env vars to calibrate to
seconds.

This is a **CMake / C++ / LibTorch** project (not a pybind extension); it builds
against the libtorch that ships inside Colab's `torch` wheel.

## 0 · Clone the repo

In [ ]:
import os, sys, subprocess, glob
REPO = "https://github.com/SniAssia/Optimized_data_loaders.git"
if not os.path.isdir("/content/Optimized_data_loaders"):
    subprocess.run(["git","clone","--depth","1",REPO,"/content/Optimized_data_loaders"], check=True)
CODE = "/content/Optimized_data_loaders/sft_libtorch_only"
assert os.path.isfile(os.path.join(CODE,"libtorch_dataloader.h")), "repo layout unexpected"
print("code:", CODE, "\nfiles:", sorted(os.listdir(CODE)))

In [ ]:
# deps: torch is preinstalled on Colab (brings libtorch + cmake prefix); add transformers/datasets for tokenizing
import importlib
need = [m for m in ["torch","transformers","datasets"] if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,"-m","pip","install","-q",*need], check=True)
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
TORCH_CMAKE = torch.utils.cmake_prefix_path
print("libtorch cmake prefix:", TORCH_CMAKE)

## 1 · Build the C++ benchmark (CMake against Colab's libtorch)

In [ ]:
# cmake is usually present on Colab; install if missing
if subprocess.run(["which","cmake"], capture_output=True).returncode != 0:
    subprocess.run([sys.executable,"-m","pip","install","-q","cmake"], check=True)

build = os.path.join(CODE, "build")
os.makedirs(build, exist_ok=True)
cfg = subprocess.run(["cmake", f"-DCMAKE_PREFIX_PATH={TORCH_CMAKE}",
                      "-DCMAKE_BUILD_TYPE=Release", ".."],
                     cwd=build, capture_output=True, text=True)
print(cfg.stdout[-1500:])
if cfg.returncode!=0: print("CMAKE STDERR:\n",cfg.stderr[-3000:]); raise RuntimeError("cmake configure failed")

mk = subprocess.run(["cmake","--build",".","--target","dataloader_libtorch","-j","2"],
                    cwd=build, capture_output=True, text=True)
print(mk.stdout[-2000:])
if mk.returncode!=0: print("BUILD STDERR:\n",mk.stderr[-4000:]); raise RuntimeError("build failed")
BIN = os.path.join(build, "dataloader_libtorch")
print("built:", BIN, "| exists:", os.path.exists(BIN))

## 2 · Tokenize REAL datasets → shards + manifest.json

Uses the repo's `tokenize_dataset.py`. `--input` is repeatable (HF name or JSONL).
First run downloads the datasets + tokenizer.

In [ ]:
DATA = os.path.join(CODE, "data")
cmd = [sys.executable, "tokenize_dataset.py",
       "--input", "tatsu-lab/alpaca",
       "--input", "HuggingFaceH4/ultrachat_200k",
       "--input", "CohereLabs/aya_dataset",
       "--output-dir", DATA,
       "--tokenizer", "facebook/opt-125m",
       "--max-seq-len", "1024",
       "--shard-size", "2000",
       "--skip-stats"]
r = subprocess.run(cmd, cwd=CODE, capture_output=True, text=True)
print(r.stdout[-2500:])
if r.returncode!=0: print("STDERR:\n",r.stderr[-3000:]); raise RuntimeError("tokenize failed")
import json
man = json.load(open(os.path.join(DATA,"manifest.json")))
print("manifest: max_seq_len",man["max_seq_len"],"| shards",len(man["shards"]))

## 3 · Run the benchmark (prints PROXY TRAINING TIME)

In [ ]:
MANIFEST = os.path.join(DATA, "manifest.json")
BATCH = 32
env = dict(os.environ, PROXY_ALPHA="1.0", PROXY_BETA="0.0")  # relative padded-token units
r = subprocess.run([BIN, MANIFEST, str(BATCH)], cwd=CODE, env=env,
                   capture_output=True, text=True)
print(r.stdout[-6000:])
if r.returncode!=0: print("STDERR:\n",r.stderr[-2000:])

## 4 · Compare to the category-scheduler pipeline

Both pipelines now print the **same proxy** (`alpha·padded_tokens + beta·batches`).
To compare fairly, run **both** on the **same datasets, same `--max-seq-len`, same
batch size**, then compare:

| metric | this (bucketing) | category pipeline |
|---|---|---|
| padded tokens | from report | from `proxy_benchmark.json` |
| padding % | from report | from `proxy_benchmark.json` |
| proxy train time | from report | from `proxy_benchmark.json` |

Lower padded-tokens / proxy-time = less wasted compute. The category pipeline's
`run_proxy.py` also prints a baseline (random batching) column, so you get a
three-way view: random baseline vs bucketing vs category scheduling.